# R4 — TextGuard v1's judge on the held-out domains (Qwen2.5-1.5B-Instruct) — Kaggle

TextGuard R4 (pre-registration PROJECT_STATUS §10g). P6's judge model and scoring code; prompts from
`judge_prompt.py` (v1, P6's prompt unchanged: whole text × change), plus the held-out questions (`heldout_questions.py`, approved 2026-10-16). Every distinct
held-out text (or unit) of real and every attack. P5's 40-prompt probe runs first and is recorded.

* Input: `r4_heldout_inputs.csv` in any attached dataset; sha256 checked.
* Output: `r4_judge_v1_kaggle_predictions.csv` + `r4_judge_v1_kaggle_run_meta.json`, appended every 20.

* **Settings (right panel → Session options):** Accelerator **GPU T4 ×2** (only cuda:0 is used),
  **Internet on**. **+ Add Input** → your dataset holding `r4_heldout_inputs.csv`.
* Then **Save Version → Save & Run All (Commit)**. The laptop can sleep; the log prints progress.
* Outputs land in `/kaggle/working` → the finished version's **Output** tab.
* `flipped`, `a1_selective`, `a2_forward`, `a3_magnitude` rows are **synthetic
  text for defence research** — not real reports.

In [ ]:
# 0. Run configuration (pre-registered, §10g) — do not change
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # one GPU (cuda:0) even on T4 x2; before torch is imported
PLATFORM = "kaggle"
INPUT_SHA256 = "cecbdc9b14086178adc1ccbaa43a5b0c9d3854952d04265a8a8ebd1301dbf231"

In [ ]:
# 1. GPU check — exactly one visible T4
import torch, transformers
assert torch.cuda.is_available(), "No GPU: Settings > Accelerator > GPU T4 x2, then Save & Run All again"
assert torch.cuda.device_count() == 1, "exactly one GPU must be visible: restart the session and run from cell 0"
assert "T4" in torch.cuda.get_device_name(0), (
    f"pre-registered GPU is a T4, got {torch.cuda.get_device_name(0)}: Settings > Accelerator > GPU T4 x2")
print(torch.cuda.get_device_name(0), "| torch", torch.__version__, "| transformers", transformers.__version__)

In [ ]:
# 2. Platform: Kaggle. No Google Drive and no browser download: outputs are written to
#    /kaggle/working and saved as this notebook's Output when the committed run finishes.
import subprocess
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip(), flush=True)
print("visible to torch:", torch.cuda.device_count(), "GPU(s) ->", torch.cuda.get_device_name(0), flush=True)

In [ ]:
# 3. Paths and inputs (one CSV for all four R4 notebooks; judge + probe rows used here)
import os, json, math, time, datetime
import pandas as pd

import glob, hashlib
OUT_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)
found = sorted(glob.glob("/kaggle/input/**/r4_heldout_inputs.csv", recursive=True))
assert found, "r4_heldout_inputs.csv not found under /kaggle/input: + Add Input > your dataset, then run again"
shas = {p: hashlib.sha256(open(p, "rb").read()).hexdigest() for p in found}
assert len(set(shas.values())) == 1, f"several different r4_heldout_inputs.csv files attached: {shas}"
INPUTS = found[0]
INPUT_SHA = shas[INPUTS]
assert INPUT_SHA == INPUT_SHA256 or "LOCAL_TEST" in globals(), (
    f"{INPUTS} has sha256 {INPUT_SHA[:16]}, expected {INPUT_SHA256[:16]}: not the exported file")
OUT = f"{OUT_DIR}/r4_judge_v1_kaggle_predictions.csv"
META = f"{OUT_DIR}/r4_judge_v1_kaggle_run_meta.json"

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_NEW_TOKENS = 24          # forecaster probe: identical to P2.4 / P5
JUDGE_NEW_TOKENS = 4         # audit-only greedy answer of the judge

df = pd.read_csv(INPUTS, keep_default_na=False, dtype=str)
probes, rows = df[df.row_type == "probe"], df[df.row_type == "judge"]
print(f"{len(rows)} judge questions + {len(probes)} probe prompts from {INPUTS}\nsaving to {OUT}")
print(rows.groupby(["domain", "question"]).size().to_string())

## Prompts — generated from `src/models/qwen_prompt.py` (probe) and `src/textguard/judge_prompt.py` (judge). Do not edit here.

In [ ]:
# Generated verbatim from src/models/qwen_prompt.py
"""Prompt construction and output parsing for Model 5 (Qwen2.5-1.5B-Instruct, early fusion).

Pure Python, no torch: this module is the **single source of truth** for what Qwen sees and
how its answer becomes a number. ``notebooks/p2_qwen_forecaster.ipynb`` embeds this file's
functions verbatim (``tests/test_qwen.py`` fails if they drift), and ``qwen_forecaster.py``
re-builds the prompts locally to check that a predictions CSV was produced by *this* prompt.

Design decisions (PROJECT_STATUS §7b):

* **Early fusion** — numbers and text in one prompt; the model decides how to combine them.
* **No dates in the numeric part.** Values are labelled ``t-L … t-1`` (relative steps), so
  the prompt never says "March 2020". This reduces Qwen recalling memorised historical values
  from pretraining. The report text is left as published and *can* contain dates — a stated
  limitation, not something we edit away (editing would change the text condition itself).
* **``text=None`` uses the identical prompt with the report block removed** — not a
  placeholder like "no report". So A (no text) is a valid, natural prompt and B − A is the
  effect of adding the report, nothing else.
* **Parsing never drops a sample.** ``parse_number`` returns ``nan`` on failure; the caller
  records ``parse_ok=False``. P4's NaN-pairing rule (PROJECT_STATUS §6a) handles the rest.
"""

from __future__ import annotations

import hashlib
import math
import re

SYSTEM = ("You are an expert time-series forecaster. "
          "You answer with a single number and nothing else.")

PERIOD = {"monthly": "month", "weekly": "week"}


def format_value(v: float) -> str:
    """Compact, lossless-enough rendering: 282558.0 -> '282558', -66029.2 -> '-66029.2'."""
    return f"{float(v):.10g}"


def build_prompt(description: str, unit: str, freq: str, lookback, text: str | None) -> str:
    """The user message. ``text`` None/blank -> the same prompt without the report block."""
    period = PERIOD[freq]
    n = len(lookback)
    lines = [f"Series: {description}.",
             f"Unit: {unit}. Frequency: {freq}.",
             "",
             f"Most recent {n} values, oldest first (t-1 is the latest {period}):"]
    lines += [f"t-{n - i}: {format_value(v)}" for i, v in enumerate(lookback)]
    if text is not None and str(text).strip():
        lines += ["",
                  "Report published before period t:",
                  '"""',
                  str(text).strip(),
                  '"""']
    lines += ["",
              f"What is the value at t, the next {period}? "
              f"Answer with one number in {unit}, no words."]
    return "\n".join(lines)


def prompt_sha(user_prompt: str) -> str:
    """Short fingerprint of a prompt, stored with each prediction for provenance."""
    return hashlib.sha256((SYSTEM + "\n\n" + user_prompt).encode("utf-8")).hexdigest()[:16]


_NUM = re.compile(r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?")
_STEP_LABEL = re.compile(r"\bt\s*[-+]\s*\d+\b", re.IGNORECASE)   # echoed "t-1" labels
_SCALE = {"thousand": 1e3, "million": 1e6, "billion": 1e9, "trillion": 1e12}


def parse_number(output: str, unit: str = "") -> float:
    """First number in Qwen's answer, in the series' unit; ``nan`` if there is none.

    Handles what a chat model typically does: thousands separators ("282,558"), a Unicode
    minus ("−58.4"), echoed step labels ("t-1: …" is skipped, not read as -1), and a scale
    word that differs from the unit ("-66.4 billion" when the unit is "million US dollars"
    -> -66400). Anything else unparseable is ``nan`` — never a guess.
    """
    if not isinstance(output, str):
        return math.nan
    s = output.replace("−", "-").replace("–", "-")
    s = _STEP_LABEL.sub(" ", s)
    s = re.sub(r"(?<=\d),(?=\d{3}\b)", "", s)                       # 282,558 -> 282558
    m = _NUM.search(s)
    if m is None:
        return math.nan
    value = float(m.group())
    word = re.match(r"\s*(thousand|million|billion|trillion)\b", s[m.end():], re.IGNORECASE)
    unit_word = re.match(r"\s*(thousand|million|billion|trillion)\b", unit, re.IGNORECASE)
    if word:
        said = _SCALE[word.group(1).lower()]
        base = _SCALE[unit_word.group(1).lower()] if unit_word else 1.0
        value *= said / base
    return value if math.isfinite(value) else math.nan

In [ ]:
# Generated verbatim from src/textguard/judge_prompt.py
"""Prompt for TextGuard's direction judge (P6) — Qwen2.5-1.5B-Instruct reads what a report claims.

Pure Python, no torch: like ``src/models/qwen_prompt.py`` this is the **single source of truth**
for what the judge sees. ``notebooks/p6_direction_judge.ipynb`` embeds this file verbatim
(``tests/test_judge.py`` fails if it drifts), and ``text_direction.py`` rebuilds every prompt
locally to check that a judge CSV was produced by *this* prompt.

Why Qwen and not the P3 rules: the fakes were written by the rules, so rules judging them would
be circular — they would recognise their own edits perfectly. The judge must be a reader that
had no part in writing the attack (PROJECT_STATUS §10b, issue #25).

Design decisions (fixed before any judge output existed, §10b):

* **The question names the forecast target's subject**, per domain, because TextGuard asks
  "what does this text claim about *my* series?". A report about another subject (e.g. a
  cross-domain text) should be answered ``none``.
* **Literal subject, not the series.** Economy's question is about the *trade deficit*, the
  quantity the reports talk about; ``text_direction.py`` maps "deficit up" to "balance down".
  Asking a 1.5B model to do that sign inversion itself would add an avoidable error source, and
  the literal answer is what the P1 human labels describe.
* **Two kinds of claim.** ``change`` — did it go up or down (all domains). ``level`` — is it
  above or below the stated national baseline (Health_US only: its reports state levels,
  P3 attacks them with the level-flip).
* **Constrained answer, scored, not parsed.** The notebook computes the probability of each of
  the three allowed answers (teacher forcing) and takes the most likely; confidence = its
  probability renormalised over the three. So there are no parse failures, and the confidence
  is the model's own, not a number it was asked to make up.
"""

from __future__ import annotations

import hashlib

JUDGE_SYSTEM = ("You read official statistical reports and say exactly what they claim. "
                "You answer with one word.")

# answer words per question kind; the order is (+1, -1, 0)
ANSWERS = {"change": ("up", "down", "none"), "level": ("above", "below", "none")}

# per domain: (question kind, subject as written in the question). Health_US asks both.
QUESTIONS = {
    "Economy": [("change", "the U.S. international trade deficit")],
    "Traffic": [("change", "travel (vehicle miles traveled) on U.S. roads compared with the "
                           "same month one year earlier")],
    "SocialGood": [("change", "the U.S. unemployment rate")],
    "Health_US": [("level", "the percentage of outpatient visits for influenza-like illness "
                            "(ILI)"),
                  ("change", "the percentage of outpatient visits for influenza-like illness "
                             "(ILI)")],
}


def build_judge_prompt(domain: str, kind: str, text: str) -> str:
    """The user message asking one question about one report."""
    subject = dict(QUESTIONS[domain])[kind]
    pos, neg, none = ANSWERS[kind]
    if kind == "change":
        question = f"According to this report, did {subject} go up or go down?"
        rule = (f'Answer "{pos}" if the report says it increased, "{neg}" if the report says '
                f'it decreased, and "{none}" if the report says it was unchanged, does not '
                "say, or is about something else.")
    else:
        question = (f"According to this report, was {subject} above or below the national "
                    "baseline?")
        rule = (f'Answer "{pos}" if the report says it was above the national baseline, '
                f'"{neg}" if the report says it was below, and "{none}" if the report says it '
                "was at the baseline, does not say, or is about something else.")
    return "\n".join(["Report:", '"""', str(text).strip(), '"""', "",
                      f"Question: {question}", rule,
                      f"Answer with one word: {pos}, {neg} or {none}."])


def judge_sha(user_prompt: str) -> str:
    """Short fingerprint of a judge prompt, stored with each answer for provenance."""
    return hashlib.sha256((JUDGE_SYSTEM + "\n\n" + user_prompt).encode("utf-8")).hexdigest()[:16]

In [ ]:
# Generated verbatim from src/textguard/heldout_questions.py
"""Judge questions for R4's held-out domains — v1's and v2's subjects for Energy and FNSPID.

Approved by Aryan 2026-10-16 and fixed in the R4 pre-registration (PROJECT_STATUS §10g) before any
judge output on either domain. **Plain dicts, no imports:** the held-out Kaggle judge notebooks
paste this file verbatim after the judge prompt module and extend it in place
(``QUESTIONS.update(HELDOUT_QUESTIONS)`` for v1, ``SUBJECTS.update(HELDOUT_SUBJECTS)`` for v2), so
v1's frozen ``judge_prompt.py`` and v2's ``judge_prompt_v2.py`` are never edited. Locally,
:func:`register` does the same.

Domain keys: ``"Energy"`` and ``"FNSPID:<TICKER>"`` (one subject per stock).

* **v1** follows v1's own convention — the claim's frame goes into the subject (as Traffic's
  "compared with the same month one year earlier"): Energy's attacked claim is week-over-week.
  Change question only; neither domain states a baseline, so no level question. POLARITY +1.
* **v2** names only the quantity; the frame is read from each unit (``units.frame``).
"""

COMPANY_NAMES = {"NFLX": "Netflix", "GOOGL": "Alphabet", "MRK": "Merck", "NVDA": "NVIDIA", "PFE": "Pfizer",
                 "JNJ": "Johnson & Johnson", "MU": "Micron", "FDX": "FedEx", "PCG": "PG&E", "LLY": "Eli Lilly"}

HELDOUT_QUESTIONS = {
    "Energy": [("change", "the U.S. national average retail price of regular gasoline compared with "
                          "the previous week")],
    **{f"FNSPID:{t}": [("change", f"the share price of {c} stock")] for t, c in COMPANY_NAMES.items()},
}

HELDOUT_SUBJECTS = {
    "Energy": "the U.S. national average retail price of regular gasoline",
    **{f"FNSPID:{t}": f"the share price of {c} stock" for t, c in COMPANY_NAMES.items()},
}


def domain_key(domain: str, series: str) -> str:
    """The judge / forecast key of a held-out window: ``Energy`` or ``FNSPID:<TICKER>``."""
    return domain if domain == "Energy" else f"{domain}:{series}"


def register() -> None:
    """Extend v1's QUESTIONS and v2's SUBJECTS in place (held-out runs only; no file is edited)."""
    from src.textguard import judge_prompt, judge_prompt_v2
    for k, v in HELDOUT_QUESTIONS.items():
        if judge_prompt.QUESTIONS.get(k, v) != v:
            raise ValueError(f"{k}: a different v1 question is already registered")
    judge_prompt.QUESTIONS.update(HELDOUT_QUESTIONS)
    judge_prompt_v2.SUBJECTS.update(HELDOUT_SUBJECTS)

# held-out questions registered in place (v1's / v2's prompt files are unchanged)
QUESTIONS.update(HELDOUT_QUESTIONS)

In [ ]:
# 4. Look at one judge prompt before spending GPU time
r = rows[rows.domain == "Energy"].iloc[0]
print(build_judge_prompt(r.domain, r.question, r.text))

In [ ]:
# 5. Load the SAME revision P5 loaded (pinned), fp16; define the probe's generate() and the judge's score()
from transformers import AutoTokenizer, AutoModelForCausalLM
REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"     # P5's revision (p5_qwen_run_meta.json) - pinned, not "latest"
P5_SETUP = "Tesla T4 | torch 2.11.0+cu130 | transformers 5.18.0 | Qwen/Qwen2.5-1.5B-Instruct@989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, torch_dtype=torch.float16,
                                             device_map="cuda").eval()

def chat_ids(system: str, user_prompt: str):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user_prompt}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt")
    if not torch.is_tensor(ids):   # transformers >= 5 returns a BatchEncoding, not a tensor
        ids = ids["input_ids"]
    return ids.to("cuda")

@torch.no_grad()
def generate(user_prompt: str, system: str = SYSTEM, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    """Greedy, batch size 1 (no padding) - identical to P2.4 / P5 for the forecaster probe."""
    ids = chat_ids(system, user_prompt)
    out = model.generate(ids, attention_mask=torch.ones_like(ids), max_new_tokens=max_new_tokens,
                         do_sample=False, temperature=None, top_p=None, top_k=None,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

ANSWER_IDS = {a: tok(a, add_special_tokens=False).input_ids for k in ANSWERS for a in ANSWERS[k]}
print("answer tokens:", ANSWER_IDS)

@torch.no_grad()
def score(user_prompt: str, answers) -> list:
    """log P(answer | judge prompt) for each allowed answer (teacher forcing, summed over tokens)."""
    ids = chat_ids(JUDGE_SYSTEM, user_prompt)
    if all(len(ANSWER_IDS[a]) == 1 for a in answers):        # one forward pass is enough
        lsm = torch.log_softmax(model(ids).logits[0, -1].float(), -1)
        return [lsm[ANSWER_IDS[a][0]].item() for a in answers]
    out = []
    for a in answers:                                         # multi-token answer: full forward each
        a_ids = torch.tensor([ANSWER_IDS[a]], device="cuda")
        logits = model(torch.cat([ids, a_ids], 1)).logits[0].float()
        lsm = torch.log_softmax(logits[ids.shape[1] - 1:-1], -1)
        out.append(lsm.gather(1, a_ids[0][:, None]).sum().item())
    return out

meta = dict(model_id=MODEL_ID, revision=REVISION, dtype="float16", batch_size=1,
            judge="scored answers (teacher forcing), confidence renormalised over 3",
            answer_tokens=ANSWER_IDS, max_new_tokens_probe=MAX_NEW_TOKENS,
            max_new_tokens_judge=JUDGE_NEW_TOKENS,
            gpu=torch.cuda.get_device_name(0), torch=torch.__version__,
            transformers=transformers.__version__, inputs=os.path.basename(INPUTS),
            platform=PLATFORM, input_path=INPUTS, input_sha256=INPUT_SHA,
            gpu_count_visible=torch.cuda.device_count(),
            started=datetime.datetime.now(datetime.timezone.utc).isoformat())
# the software setup that produces every row of this run — same format as qwen_forecaster.setup_id
SETUP = (f"{meta['gpu']} | torch {meta['torch']} | transformers {meta['transformers']} | "
         f"{MODEL_ID}@{REVISION} | {PLATFORM}")
meta["setup"] = SETUP
json.dump(meta, open(META, "w"), indent=1)
print("SETUP:", SETUP, flush=True)

In [ ]:
# 6. Reproducibility probe: P5's 40 forecasting prompts must reproduce P2.4's raw answers exactly
mismatch = []
for r in probes.itertuples():
    raw = generate(build_prompt(r.description, r.unit, r.freq, json.loads(r.lookback), r.text or None))
    if raw != r.expected_raw:
        mismatch.append((r.domain, r.sample_id, r.condition, r.expected_raw, raw))
PROBE_OK = len(probes) == 40 and not mismatch
print(f"probe: {len(probes) - len(mismatch)}/{len(probes)} identical to P2.4 ->",
      "SAME SETUP CLASS AS P5 (certified)" if PROBE_OK
      else "DIFFERENT SETUP CLASS from P5 - the judge still runs on one setup; this is recorded")
print("setup:", SETUP, "| P5 (Colab):", P5_SETUP, flush=True)
for m in mismatch[:10]:
    print("  ", m)
meta.update(probe_n=len(probes), probe_mismatches=len(mismatch), probe_ok=PROBE_OK,
            same_setup_string_as_p5=SETUP == P5_SETUP)
json.dump(meta, open(META, "w"), indent=1)

In [ ]:
# 7. Main loop — resumable. Appends every 20 questions; a disconnect loses at most 20 rows.
#    Re-run this cell (after cells 1-6) to continue where it stopped.
COLS = ["domain", "question", "prompt_sha", "answer", "confidence", "p_pos", "p_neg", "p_none",
        "logp_pos", "logp_neg", "logp_none", "raw_output", "setup"]
done = set()
if os.path.exists(OUT):
    prev = pd.read_csv(OUT, keep_default_na=False, dtype=str)
    other = set(prev.setup) - {SETUP}
    assert not other, (f"{OUT} holds rows from another software setup {other}; one file = one setup. "
                       "Rename or delete it and run again from cell 1.")
    done = set(prev.prompt_sha)
    print(f"resuming: {len(done)} answers already saved")
else:
    pd.DataFrame(columns=COLS).to_csv(OUT, index=False)

todo = [r for r in rows.itertuples() if r.prompt_sha not in done]
print(f"===== MAIN LOOP | {SETUP} =====", flush=True)
print(f"{len(todo)} questions to answer", flush=True)
meta.update(rows_expected=len(rows))
json.dump(meta, open(META, "w"), indent=1)
buf, t0 = [], time.time()
for k, r in enumerate(todo, 1):
    up = build_judge_prompt(r.domain, r.question, r.text)
    assert judge_sha(up) == r.prompt_sha, "judge prompt differs from the exported one"
    answers = ANSWERS[r.question]
    lp = score(up, answers)
    m = max(lp)
    p = [math.exp(x - m) for x in lp]
    p = [x / sum(p) for x in p]
    best = max(range(3), key=lambda i: (p[i], -i))    # ties -> earlier answer (never happens in practice)
    buf.append(dict(domain=r.domain, question=r.question, prompt_sha=r.prompt_sha,
                    answer=answers[best], confidence=p[best], p_pos=p[0], p_neg=p[1], p_none=p[2],
                    logp_pos=lp[0], logp_neg=lp[1], logp_none=lp[2],
                    raw_output=generate(up, system=JUDGE_SYSTEM, max_new_tokens=JUDGE_NEW_TOKENS),
                    setup=SETUP))
    if len(buf) >= 20 or k == len(todo):
        pd.DataFrame(buf, columns=COLS).to_csv(OUT, mode="a", header=False, index=False)
        buf = []
    if k % 100 == 0 or k == len(todo):
        el = time.time() - t0
        print(f"{k}/{len(todo)}  {el / 60:.1f} min elapsed, ~{el / k * (len(todo) - k) / 60:.1f} min left",
              flush=True)

In [ ]:
# 8. Completeness and answer counts only — judge accuracy is P6b, not here
res = pd.read_csv(OUT, keep_default_na=False, dtype=str)
missing = set(rows.prompt_sha) - set(res.prompt_sha)
assert not missing, f"{len(missing)} questions missing: re-run cell 7"
assert not res.prompt_sha.duplicated().any()
assert set(res.setup) == {SETUP}, "rows from more than one software setup"
print(f"{len(res)} answers from one setup")
print(res.groupby(["domain", "question", "answer"]).size().unstack(fill_value=0).to_string())
meta.update(finished=datetime.datetime.now(datetime.timezone.utc).isoformat(), rows=len(res))
json.dump(meta, open(META, "w"), indent=1)

In [ ]:
# 9. Outputs — in /kaggle/working; after the committed run: the version's Output tab -> download both
for f in (OUT, META):
    print(f"{f}  ({os.path.getsize(f) / 1024:.0f} KB)", flush=True)
print("RUN FINISHED:", len(res), "answers | probe", f"{meta['probe_n'] - meta['probe_mismatches']}/{meta['probe_n']}",
      flush=True)